### Setting the topology

In a YAML file, define the parameters and topology for your test in a similar manner as the following:
```yaml
topology:
  name: "g5k_mcast_eval"
  wall_time: "2hr"
  relay_nodes: false # whether to add one relay machine in each cluster
  netns_per_client: 5 # number of network namespaces to run on each client
  # see the possible frrouting version at https://deb.frrouting.org/
  frrouting_version: "frr-10.4"
  router_template: "base_router_config_ospf.frr" # path to the router configuration template

  server:
    cluster: "chirop" # Lille
    nodes: 1
    # node: "chirop-5.lille.grid5000.fr"   # optional: pin a specific machine

  # each site has one router + num_clients clients and one relay,
  # all reserved in the given cluster. 
  sites:
    - name: nancy
      cluster: gros
      num_clients: 5
    - name: rennes
      cluster: parasilo
      num_clients: 5
    - name: nantes
      cluster: ecotype
      num_clients: 5
    - name: lyon
      cluster: nova
      num_clients: 5

  # links are established between routers in different clusters
  # GRE tunnels are established between the two routers, with OSPF running over it.
  # endpoints must be router_server or router_client_CLIENT-CLUSTER-ID
  links:
    - [router_server, router_client_0]             # src -> nancy
    - [router_client_0, router_client_1]           # nancy -> rennes
    - [router_client_0, router_client_3]           # nancy -> lyon
    - [router_client_0, router_client_2]           # rennes -> nantes
``` 


In [ ]:
!pip install enoslib ipywidgets==8.1.5 fabric --break-system-packages


### Setting up the experiment
Once you have your `topology.yaml` file, you can create the `G5KExpe` class which will handle most things for you.

In [ ]:
from g5k_eval import G5KExpe

experiment = G5KExpe(
    # change the path to point to your topology yaml file
    topology_conf="./mcast_eval.yaml",
    #
    # other parameters exist:
    # g5k_conf_file_loc points to your .python-grid5000.yaml file which contains your grid5000 credentials, by default it is in `~/` (so `/home/USERNAME`)
    # g5k_conf_file_loc=".python-grid5000.yaml"
    #
    # job_type should be deploy, but you may need it to be different
    # job_type="deploy"
    #
    # os_env_name defines the OS environement that is deployed on the machines
    # by default it is debian12 with NFS, however you can find the entire list at https://www.grid5000.fr/w/Getting_Started#:~:text=On%20Grid%275000%20reference%20environments
    # Make sure to pick debian to ensure that the packages are properly installed
    # os_env_name="debian12-nfs"
    #
    # You can configure the number of ansible forks used, ansible's default is 5, meaning that it'll run commands on at most 5 host at once
    # in this framework the default is 25 to make use of more parallelism, however, increasing this value will consume more resources (especially memory)
    # setting the number of forks to 200 will consume around 25 GB of memory but will allow ansible to perform operations on 200 hosts at the same time
    # ansible_forks=25
)

# you should always follow grid5000's usage policy (see https://www.grid5000.fr/w/Grid5000:UsagePolicy)
# this method simply checks that the job you are trying to start will not cross the day-night boundary.
# If it does, it'll warn you. You can always comment this out if you wish...
experiment.usage_policy_check()

provider = experiment.setup_enoslib_conf()


### Reserving resources
Now that G5K is setup, we can create the experiment's reservation by defining the number of machines of each role and in each cluster.

Once done, we proceed with the actual reservation of the machines. Be aware that this step may take some time (minimum 5 minutes). This is due to the deployment of the VM image. 

**Don't forget to run "ssh-add KEY_PATH" to allow ansible to connect using your ssh key**

In [ ]:
experiment.reserve_res(provider)

#### Setting up interfaces, IP subnets, and Network namespaces

In [ ]:
experiment.setup_interfaces()
experiment.assign_node_ips()
experiment.netns_setup_macvlan()

### Setting up GRE tunnels between routers in different clusters

This step will create GRE tunnels between each pair of routers as defined in the topology file. The endpoints of the tunnels use the production IP of the nodes.

In [ ]:
experiment.setup_gre_tunnels()

#### FRRouting setup

With GRE tunnels setup between rotuers, we can now configure and start FRRouting. The frr configuration template defined in the topology file will be used as a base.

In [ ]:
experiment.frrouting_setup()
experiment.setup_default_routes()

### Upload binary files over to nodes

We build the executables locally first

In [ ]:
!cd ../../../g5k_mcast_eval && cargo build --release

Then we push them to the nodes

In [ ]:
import enoslib as en

experiment.push_binaries(
    # TODO: change these paths with the path to your binaries and certificates
    bin_dir="../../../g5k_mcast_eval/target/release",
    cert_dir="../../../g5k_mcast_eval",
)

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=experiment.roles,
)
print("errors: " + str([out.stderr for out in res.filter(status=en.STATUS_FAILED)]))

### Running the relay experiment

Experiment.py provides some basic blocks that should (ideally) allow you to define your own custom experiments.
Below you will find the code for the evaluation of two types of Flexicast QUIC relays, this should hopefully provide enough information.


In [ ]:
import concurrent.futures
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
import time

import enoslib as en

from g5k_eval.experiment import (
    EvalConfig,
    MetricSpec,
    collect_results,
    run_eval,
)
from g5k_eval.remote import (
    run_cmd_bg_enos,
    run_cmd_ssh_parallel,
    send_pkill_hosts,
    ssh_bg,
)


@dataclass
class RunConfig:
    additional_data_size: int
    test_length: int


@dataclass
class CatEvalConfig(EvalConfig):
    ready_sleep_clients: int = 2
    post_test_buffer: int = 3
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    remote_log_root: str = "/tmp/logs"
    num_ns_per_client: int = experiment.topology.netns_per_client
    cc_algo: str = "disabled"
    fallback_delay: int = 10000
    server_cpus: str = "0-1"  # taskset -c range for server


# define the results to extract from the client logs, one csv output file is emitted for each metric
METRICS = [
    MetricSpec(key="LATENCY", column="y_LATENCY"),
    #  you can add more result types here, e.g.:
    # MetricSpec(key="THROUGHPUT", column="y_THROUGHPUT"),
]

We can now define specific tests based on our `RunConfig`.

For the network categorization test, we simply send one packet of increasing size, and we wait until it has been received by all clients.

In [ ]:
def categorization_matrix():
    return [
        RunConfig(additional_data_size=sz, test_length=10)
        for sz in (1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000)
    ]

To enable us to have graphs that show certain metrics per cluster, we need to pass in a list of the cluster names and their subnets to the clients. Here we construct the lists to pass to the clients

In [ ]:
# ---------- cluster names & subnets ----------
# the subnets are in experiment.networks, but i need the subnets keyed by their index and not their cluster
site_subnets = {
    site.name: {
        "cluster": site.cluster,
        "num_clients": site.num_clients,
        "subnet": str(
            experiment.networks[f"subnet_client_{i}"][0].network
        ),  # "10.x.y.0/22"
    }
    for i, site in enumerate(experiment.topology.sites)
}
server_subnet = str(experiment.networks["subnet_server"][0].network)

cluster_names = [site.name for site in experiment.topology.sites]
cluster_subnets = [info["subnet"] for info in site_subnets.values()]

print(f"site subnets:  {site_subnets}")
print(f"server subnet: {server_subnet}")

Now that the experiment is defined, we need to specify the commands that will be ran on the nodes.


In [ ]:
# ---------- command builders ----------
def server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--test-mode --flexicast --fc-timer 0 --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo}"
        f"--additional-data-size {rc.additional_data_size} --test-start-ts {sleep_deadline_ts}"
    )


# IMPORTANT NOTE: since we have multiple network namespaces defined on each client machine,
# we can run processes in these namespaces using the naming scheme "client-$NS_IDX" (with NS_IDX going from the number of 0 to NSs)
def client_loop_cmd(cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts):
    qlog = f"{run_dir}/qlog/client"
    return f"""
mkdir -p {run_dir}/client
mkdir -p {qlog}
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    ip netns exec $NS_NAME env RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} --test-mode \\
        --test-start-ts {sleep_deadline_ts} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} \\
         {" ".join(f"--cluster-names={name}" for name in cluster_names)} \\
         {" ".join(f"--cluster-subnets={subnet}" for subnet in cluster_subnets)} \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


# ---------- one run of the relay experiment ----------
def run_once(cfg, rc, run_index, test_name):
    """Run one iteration: start the server, start the clients in
    their namespaces, wait for the test to finish, then collect the results."""
    roles_dict = experiment.roles
    node_ips = experiment.node_ips

    server_ip = node_ips["server"][0]
    run_id = f"run_t{rc.test_index}_sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts

    # create the dirs on all of the hosts
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/client {run_dir}/qlog",
        all_hosts,
    )

    # make sure all programs are stopped
    send_pkill_hosts(all_hosts, ["server", "client"])
    time.sleep(1)

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=5)
    sleep_deadline_ts = sleep_deadline.timestamp()

    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )

    time.sleep(cfg.ready_sleep_relay)

    # start all clients in // to make them start kinda at the same time
    def _start_client(node_id, h):
        cmd = client_loop_cmd(cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts)
        ssh_bg(
            cmd,
            h,
            stdout=f"{run_dir}/client/loop_{node_id}.stdout",
            stderr=f"{run_dir}/client/loop_{node_id}.stderr",
        )

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=max(1, len(client_hosts))
    ) as ex:
        list(ex.map(lambda p: _start_client(*p), list(enumerate(client_hosts))))

    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "client"])

    time.sleep(1)

    results = collect_results(cfg, client_hosts, run_dir, test_name, METRICS)
    cpu_samples = (
        collect_cpuload(cfg, roles_dict["server"][0], run_dir, test_name)
        if cfg.monitor_cpu
        else []
    )
    return results, cpu_samples

Lauching the test

In [ ]:
N_RUNS = 1

cfg = CatEvalConfig(n_runs=N_RUNS, monitor_cpu=False)
now = datetime.now().strftime("%d-%m-%H-%M%p")

test_name = f"categorization_{now}"
matrix = categorization_matrix()


def row_fields(rc):
    # columns identifying each run in the result CSVs
    return {
        "test_index": rc.test_index,
        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
    }


run_eval(
    matrix,
    cfg,
    run_once=run_once,
    test_name=test_name,
    row_fields=row_fields,
    metrics=METRICS,
)

### Downloading SQLOGs from server and relay
# TODO: modify this to download QLOGs from clients instead of server and relays

In [ ]:
import subprocess
from pathlib import Path

# uses cfg, matrix, test_name and N_RUNS from the launching cell above
local_base = Path(f"./sqlogs/{test_name}")

for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

        remote_qlog_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}/qlog/server"
        local_dir = local_base / run_id / "server"
        local_dir.mkdir(parents=True, exist_ok=True)

        # download the sqlogs from the server only
        for node in experiment.roles["server"]:
            host = node.address
            print(f"downloading sqlogs from {host} to {remote_qlog_dir}")
            subprocess.run(
                [
                    "rsync",
                    "-az",
                    # "-o LogLevel=ERROR",
                    "--include=*.sqlog",
                    "--exclude=*",
                    f"root@{host}:{remote_qlog_dir}/",
                    f"{local_dir}/",
                ],
                check=False,
            )

print(f"results: {local_base}")

### Merging SQLOG files together

In [ ]:
from pathlib import Path
import sys
import tempfile
import re
import json
import csv


def merge_sqlogs(files, output):
    control_chars = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")

    with open(output, "w") as out:
        for i, f in enumerate(files):
            with open(f) as src:
                first = True
                for j, line in enumerate(src):
                    if j == 0 and i > 0:
                        # if i > 0, then we wrote the header once already, so now skip the headers (first lines of sqlog files: j==0)
                        continue

                    line = line.rstrip() + "\n"
                    line = control_chars.sub("", line)
                    out.write(control_chars.sub("", line))


def extract_path_acks(sqlog, csv_out):

    # we need to get the path_ack lengths from the sqlogs
    # go through the merged sqlog, and append to a csv file the time and length of each path_ack we see
    with open(sqlog) as src, open(csv_out, "w", newline="") as out:
        writer = csv.writer(out)
        writer.writerow(["time", "length"])

        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":12.632589,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":4},"raw":{"length":1155,"payload_length":1138},"frames":[{"frame_type":"path_ack","path_identifier":0,"ack_delay":0.085,"acked_ranges":[[3,3]]},{"frame_type":"path_new_connection_id","path_id":1,"sequence_number":0,"retire_prior_to":0,"connection_id_length":16,"connection_id":"ab44f5dde5157072f203e53c8d76836d","stateless_reset_token":"536abfd4dc2107e6e1188cbba08f4ce1"},{"frame_type":"padding","payload_length":1071}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []

                for frame in frames:
                    if frame.get("frame_type") == "path_ack":
                        # if we do have a path_ack frame (migth contain other stuff), get the length
                        length = event.get("data", {}).get("raw", {}).get("length")

                        if length is not None:
                            writer.writerow([event.get("time"), length])


local_base = Path(f"./sqlogs/{test_name}")

for relay_test in ["none", "RELAY", "APP_RELAY"]:

    trace_files = []
    for run_conf in matrix:
        if run_conf.relay_version != relay_test:
            continue

        for run_index in range(N_RUNS):
            run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

            server_dir = local_base / run_id / "server"

            for file in sorted(server_dir.glob("server-server-*.sqlog")):
                trace_files.append(file)

    if not trace_files:
        print(f"no files found for {relay_test}")
        continue

    print(f"relay={relay_test}: {len(trace_files)} trace files")

    temp_dir = Path(tempfile.mkdtemp(prefix="ackrate_"))
    merged_log = temp_dir / f"merged_{relay_test}.sqlog"

    merge_sqlogs(trace_files, merged_log)

    merged_csv = Path(f"./npf-out/ack_rate_{test_name}") / f"{relay_test}.csv"
    merged_csv.parent.mkdir(parents=True, exist_ok=True)

    extract_path_acks(merged_log, merged_csv)
    print(f"path_ack csv: {merged_csv}")

### Graphing the results
# TODO


In [ ]:
import subprocess
from pathlib import Path

INSET_GRAPHS = True
NO_TITLE = True
out_path = f"./graphs/{test_name}/"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./relay_graphs.py",
        f"./npf-out/{test_name}.csv",  # input csv paht
        out_path,  # out path
        test_name,
        f"./npf-out/ack_rate_{test_name}/",  # ack_rate_path
        f"./npf-out/{test_name}_cpu.csv",  # cpu_csv_path
        *(["--inset"] if INSET_GRAPHS else []),
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

#### Compressing the csv results

In [ ]:
import subprocess

# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

Opposite code to unarchive the results, in order to regenerate graphs if needed

In [ ]:
import subprocess
from pathlib import Path

# og_name = "sserv_2thr_latency_test_large_relay_topo_26-04-21-39PM_481mbps"
test_name = "serv_2thr_latency_test_large_relay_topo_26-04-21-39PM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

#### Deleting log files from all clusters

In [ ]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"
matrix = categorization_matrix()
for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=experiment.roles["client"] + experiment.roles["server"],
    )

print("done deleting sqlog files")

## Important: Stopping the current booking
Always, always stop your booking if you are done earlier.

In [ ]:
experiment.stop_reservation()